# Аналіз результатів симуляцій БпЛА та НРК

Обробка єдиного файлу датасету `som_navigation_benchmark_410.jsonl.gz` (410 повторень), жорсткий відбір допустимих моделей за критеріями п. 3.5 (табл. 3) та побудова однопанельних наукових графіків і таблиць для підрозділу 4 статті.

In [ ]:
import os, gzip, json, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter

plt.style.use("default")

COLORS = {
    "multimodal": "C0",
    "text": "C1",
    "heuristic": "C7",
}

FIG_DIR = "/content/figures"
TAB_DIR = "/content/tables"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

def fmt_num(val, dec=1):
    return f"{np.round(val, dec):.{dec}f}".replace(".", ",")

def comma_axis(dec=None):
    def _fmt(v, _pos):
        if dec is not None:
            return f"{np.round(v, dec):.{dec}f}".replace(".", ",")
        if float(v).is_integer():
            return f"{int(round(v))}"
        return f"{v:g}".replace(".", ",")
    return FuncFormatter(_fmt)

def save_fig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, f"{name}.png"), dpi=300, bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, f"{name}.pdf"), bbox_inches="tight")
    plt.show()
    plt.close(fig)

def mode_handles(include_heuristic=True, include_pareto=False):
    items = [
        ("multimodal", "Мультимодальний (MM)"),
        ("text", "Текстовий (txt)"),
    ]
    if include_heuristic:
        items.append(("heuristic", "Базова евристика"))
    handles = [patches.Patch(color=COLORS[k], label=lbl) for k, lbl in items]
    if include_pareto:
        handles.append(Line2D([], [], marker="*", color="none", markerfacecolor="black", markeredgecolor="black", markersize=9, label="Множина Парето"))
    return handles

def ylbl(row):
    return f"★ {row['label']}" if bool(row.get("pareto", False)) else row["label"]


## 1. Завантаження єдиного файлу датасету та парсинг метрик і траєкторій

In [ ]:
DATASET_PATH = "/content/som_navigation_benchmark_410.jsonl.gz"

def open_dataset(path):
    if path.endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8")
    return open(path, "r", encoding="utf-8")

def _pick(d, jev, key1, key2, default=None):
    val = d.get(key1)
    if val is None:
        val = jev.get(key2, default)
    return val

rows = []
run1_trajectories = {}
with open_dataset(DATASET_PATH) as f:
    for line in f:
        if not line.strip():
            continue
        rec = json.loads(line)
        dom = rec["domain"]
        run_name = rec["run_name"]
        cfg = rec["config"]
        d = rec["metrics"]
        if run_name == "run_1":
            run1_trajectories[(dom, cfg)] = rec["trajectory"]
        model = d["model"]
        vision = bool(d.get("vision", False))
        mode = "heuristic" if model == "heuristic" else ("multimodal" if vision else "text")
        name = model.split("/")[-1]
        provider = "Базовий" if model == "heuristic" else model.split("/")[0]
        label = "Базова евристика" if mode == "heuristic" else f"{name} ({'MM' if vision else 'txt'})"
        jev = d.get("jev") if isinstance(d.get("jev"), dict) else {}

        t50 = _pick(d, jev, "median_latency_s", "median_latency_s")
        t90 = _pick(d, jev, "p90_latency_s", "p90_latency_s")
        n_calls = _pick(d, jev, "api_calls", "calls", 0)
        n_err = _pick(d, jev, "api_errors", "errors", 0)
        n_tok = _pick(d, jev, "total_tokens", "tokens", 0)

        if dom == "drone":
            x_val = float(d["max_x_m"])
            pb_val = float(bool(d.get("crossed_barrier", False))) * 100.0
            s_val = float(d.get("crashed_at_s") is None) * 100.0
            t_val = float(d["flew_s"])
            ra_val = float(d.get("steps_jev_acted_pct", 0.0))
            rr_val = float(d.get("steps_reflex_pct", 0.0))
            r_heu, r_clr = np.nan, np.nan
        else:
            x_val = float(d["max_x"])
            pb_val = float(x_val > 29.0) * 100.0
            s_val = np.nan
            t_val = float(d["final_t"])
            lt = d.get("layer_ticks", {})
            tot = sum(lt.values()) or 1
            ra_val = 100.0 * lt.get("model", 0) / tot
            rr_val = np.nan
            r_heu = 100.0 * lt.get("heuristic", 0) / tot
            r_clr = 100.0 * lt.get("clear", 0) / tot

        rows.append({
            "domain": dom,
            "run": run_name,
            "config": cfg,
            "provider": provider,
            "model": model,
            "name": name,
            "mode": mode,
            "vision": vision,
            "label": label,
            "X": x_val,
            "Pb": pb_val,
            "V": float(d.get("target_visible_pct", np.nan)),
            "D": float(d.get("mean_standoff_m", np.nan)),
            "C": int(d.get("collisions", 0)),
            "S": s_val,
            "T": t_val,
            "Ra": ra_val,
            "Rr": rr_val,
            "R_heu": r_heu,
            "R_clr": r_clr,
            "T50": float(t50) if t50 is not None else np.nan,
            "T90": float(t90) if t90 is not None else np.nan,
            "n": int(n_calls or 0),
            "Ne": int(n_err or 0),
            "U": int(n_tok or 0),
        })

df_runs = pd.DataFrame(rows)

def aggregate(df_dom):
    cols = ["config", "provider", "model", "name", "mode", "vision", "label"]
    g = df_dom.groupby(cols, as_index=False).agg(
        X=("X", "mean"), X_sd=("X", "std"),
        Pb=("Pb", "mean"),
        V=("V", "mean"), V_sd=("V", "std"),
        D=("D", "mean"), D_sd=("D", "std"),
        C=("C", "mean"), C_sd=("C", "std"),
        S=("S", "mean"),
        T=("T", "mean"), T_sd=("T", "std"),
        Ra=("Ra", "mean"), Ra_sd=("Ra", "std"),
        Rr=("Rr", "mean"), Rr_sd=("Rr", "std"),
        R_heu=("R_heu", "mean"), R_heu_sd=("R_heu", "std"),
        R_clr=("R_clr", "mean"), R_clr_sd=("R_clr", "std"),
        T50=("T50", "mean"), T50_sd=("T50", "std"),
        T90=("T90", "mean"), T90_sd=("T90", "std"),
        n=("n", "mean"), n_sd=("n", "std"),
        Ne=("Ne", "mean"), Ne_sd=("Ne", "std"),
        Ne_total=("Ne", "sum"),
        U=("U", "mean"), U_sd=("U", "std"),
        valid_lat_runs=("T50", lambda s: int(s.notna().sum())),
    )
    g["low_lat_validity"] = (g["mode"] != "heuristic") & ((g["valid_lat_runs"] < 5) | (g["n"] < 5))
    return g.sort_values(["X", "C"], ascending=[False, True]).reset_index(drop=True)

df_uav = aggregate(df_runs[df_runs["domain"] == "drone"])
df_ugv = aggregate(df_runs[df_runs["domain"] == "rover"])
uav_max_x = float(df_uav["X"].max())
ugv_max_x = float(df_ugv["X"].max())

def make_paired(df_dom):
    tx = df_dom[df_dom["mode"] == "text"].set_index("model")
    mm = df_dom[df_dom["mode"] == "multimodal"].set_index("model")
    return tx.join(mm, how="inner", lsuffix="_tx", rsuffix="_mm").reset_index()

df_pair_uav = make_paired(df_uav)
df_pair_ugv = make_paired(df_ugv)
print(f"Runs: {len(df_runs)} | UAV configs: {len(df_uav)} | UGV configs: {len(df_ugv)}")


## 2. Жорсткий фільтр допустимості та множина Парето
Перед побудовою графіків відсікаються нестійкі, аварійні та ненадійні конфігурації за однаковими жорсткими умовами допустимості з п. 3.5:
* **БпЛА:** $S = 100\,\%$ (жодного падіння), $P_b = 100\,\%$ (подолання балки в усіх 5 із 5 повторень), $N_e \le 1,0$ од./повторення.
* **НРК:** $P_b = 100\,\%$ (прорив картонної стіни в усіх 5 із 5 повторень), $N_e \le 1,0$ од./повторення.
* **Крос-доменна вибірка:** моделі, які проходять жорсткий фільтр **одночасно на обох платформах**.

Саме ці відібрані допустимі конфігурації (плюс «Базова евристика» як еталон) використовуються на всіх основних графіках нижче.

In [ ]:
def mark_pareto(df, specs):
    flags = []
    for i, r in df.iterrows():
        dom = False
        for j, o in df.iterrows():
            if i == j:
                continue
            ge = all((o[k] >= r[k] if d == 1 else o[k] <= r[k]) for k, d in specs)
            gt = any((o[k] > r[k] if d == 1 else o[k] < r[k]) for k, d in specs)
            if ge and gt:
                dom = True
                break
        flags.append(not dom)
    return pd.Series(flags, index=df.index)

df_uav["feasible"] = (df_uav["mode"] != "heuristic") & (df_uav["S"] == 100.0) & (df_uav["Pb"] == 100.0) & (df_uav["Ne"] <= 1.0)
df_ugv["feasible"] = (df_ugv["mode"] != "heuristic") & (df_ugv["Pb"] == 100.0) & (df_ugv["Ne"] <= 1.0)

df_uav["pareto"] = False
df_ugv["pareto"] = False
df_uav.loc[df_uav["feasible"], "pareto"] = mark_pareto(df_uav[df_uav["feasible"]], [("X", 1), ("C", -1), ("T50", -1)])
df_ugv.loc[df_ugv["feasible"], "pareto"] = mark_pareto(df_ugv[df_ugv["feasible"]], [("X", 1), ("C", -1), ("T50", -1)])

som_uav = df_uav[df_uav["mode"] != "heuristic"].copy()
som_ugv = df_ugv[df_ugv["mode"] != "heuristic"].copy()
uav_feasible = df_uav[df_uav["feasible"]].copy().reset_index(drop=True)
ugv_feasible = df_ugv[df_ugv["feasible"]].copy().reset_index(drop=True)

uav_sel = pd.concat([uav_feasible, df_uav[df_uav["mode"] == "heuristic"]]).reset_index(drop=True)
ugv_sel = pd.concat([ugv_feasible, df_ugv[df_ugv["mode"] == "heuristic"]]).reset_index(drop=True)

df_cross = df_uav.merge(
    df_ugv,
    on=["provider", "model", "name", "mode", "vision", "label"],
    suffixes=("_uav", "_ugv"),
)
df_cross["uav_pct"] = 100.0 * df_cross["X_uav"] / uav_max_x
df_cross["ugv_pct"] = 100.0 * df_cross["X_ugv"] / ugv_max_x
df_cross["score"] = 0.5 * (df_cross["uav_pct"] + df_cross["ugv_pct"])
df_cross["feasible_both"] = df_cross["feasible_uav"] & df_cross["feasible_ugv"]

cross_feasible = df_cross[df_cross["feasible_both"]].sort_values("score", ascending=False)
cross_sel = pd.concat([cross_feasible, df_cross[df_cross["mode"] == "heuristic"]]).reset_index(drop=True)
som_cross = df_cross[df_cross["mode"] != "heuristic"].copy()

filter_report = pd.DataFrame([
    ("БпЛА (SOM)", len(som_uav), int((som_uav["S"] < 100.0).sum()), int((som_uav["Pb"] < 100.0).sum()), int((som_uav["Ne"] > 1.0).sum()), len(uav_feasible), int(df_uav["pareto"].sum())),
    ("НРК (SOM)", len(som_ugv), "—", int((som_ugv["Pb"] < 100.0).sum()), int((som_ugv["Ne"] > 1.0).sum()), len(ugv_feasible), int(df_ugv["pareto"].sum())),
    ("Обидві платформи", len(som_cross), int((som_cross["S_uav"] < 100.0).sum()), int(((som_cross["Pb_uav"] < 100.0) | (som_cross["Pb_ugv"] < 100.0)).sum()), int(((som_cross["Ne_uav"] > 1.0) | (som_cross["Ne_ugv"] > 1.0)).sum()), len(cross_feasible), "—"),
], columns=["Платформа", "Усього SOM", "Відсіяно S<100%", "Відсіяно по Pb", "Відсіяно Ne>1", "Допустимих", "У множині Парето"])

display(filter_report)
print("БпЛА допустимі:", uav_feasible["label"].tolist())
print("БпЛА Парето (X+, C-, T50-):", df_uav[df_uav["pareto"]]["label"].tolist())
print("НРК допустимі:", ugv_feasible["label"].tolist())
print("НРК Парето (X+, C-, T50-):", df_ugv[df_ugv["pareto"]]["label"].tolist())
print("Крос-доменні допустимі:", cross_feasible["label"].tolist())


## 3. Основні графіки для Розділу 4: БпЛА (п. 4.2)
* **Рис. 4.1** — Супровід цілі $V$ та просування $X$ для допустимих конфігурацій БпЛА та «Базової евристики» (зірочкою `★` позначено моделі з множини Парето).
* **Рис. 4.2** — Середня кількість аварійних зіткнень $C$ допустимих конфігурацій БпЛА.
* **Рис. 4.3** — Зв’язок медіанної затримки $T_{50}$ і зіткнень $C$ по всіх 39 валідних конфігураціях SOM БпЛА (зірочки — множина Парето, кола — інші допустимі, хрестики — відсіяні).
* **Рис. 4.3б** — Інтервали обчислювальної затримки $[T_{50};\, T_{90}]$, виклики $n$ і витрата токенів $U$ для допустимих конфігурацій БпЛА.
* **Рис. 4.3в** — Активність тактичного контуру SOM ($R_a$) та рефлексу безпеки ($R_r$) на БпЛА.
* **Рис. 4.3г** — Траєкторії моделей із множини Парето БпЛА, базової евристики та відсіяної моделі у горизонтальній площині $XY$.
* **Рис. 4.3д** — Вертикальний профіль польоту $XZ$ (переліт низьких балок на $x = 19$ та $44$ м).

In [ ]:
sub_v = uav_sel.sort_values("V", ascending=False).reset_index(drop=True)
yv = np.arange(len(sub_v))

fig, ax = plt.subplots(figsize=(9.0, 6.8))
ax.barh(yv, sub_v["V"], color=[COLORS[m] for m in sub_v["mode"]])
for i, r in sub_v.iterrows():
    lbl_txt = f"{fmt_num(r['V'], 1)} % ({fmt_num(r['X'], 1)} м)"
    ax.text(r["V"] + 1.5, i, lbl_txt, va="center", fontsize=8.4)
ax.set_yticks(yv, [ylbl(r) for _, r in sub_v.iterrows()])
ax.invert_yaxis()
ax.set_xlim(0, 134)
ax.set_xticks([0, 20, 40, 60, 80, 100])
ax.xaxis.set_major_formatter(comma_axis(0))
ax.set_xlabel("Утримання цілі в полі зору $V$, %")
ax.set_title("БпЛА: супровід цілі $V$", pad=52)
ax.grid(axis="x", alpha=0.3)
ax.legend(handles=mode_handles(True, True), loc="lower center", bbox_to_anchor=(0.5, 1.015), ncol=2, frameon=True)
save_fig(fig, "fig4_1_uav_tracking")

sub_c = pd.concat([uav_feasible.sort_values(["C", "X"], ascending=[True, False]), df_uav[df_uav["mode"] == "heuristic"]]).reset_index(drop=True)
yc = np.arange(len(sub_c))

fig, ax = plt.subplots(figsize=(8.8, 6.8))
ax.barh(yc, sub_c["C"], color=[COLORS[m] for m in sub_c["mode"]])
for i, r in sub_c.iterrows():
    lbl_c = f"{fmt_num(r['C'], 1)} ({fmt_num(r['X'], 1)} м)"
    ax.text(r["C"] + 0.06, i, lbl_c, va="center", fontsize=8.4)
ax.set_yticks(yc, [ylbl(r) for _, r in sub_c.iterrows()])
ax.invert_yaxis()
ax.set_xlim(0, 4.45)
ax.xaxis.set_major_formatter(comma_axis(1))
ax.set_xlabel("Середня кількість унікальних зіткнень $C$, од.")
ax.set_title("БпЛА: зіткнення $C$", pad=52)
ax.grid(axis="x", alpha=0.3)
ax.legend(handles=mode_handles(True, True), loc="lower center", bbox_to_anchor=(0.5, 1.015), ncol=2, frameon=True)
save_fig(fig, "fig4_2_uav_collisions")


In [ ]:
valid_uav = som_uav[~som_uav["low_lat_validity"]].copy()

fig, ax = plt.subplots(figsize=(13.4, 8.4))
for mode_key, lbl in [("multimodal", "MM"), ("text", "txt")]:
    sub_par = valid_uav[(valid_uav["mode"] == mode_key) & (valid_uav["pareto"])]
    sub_ok = valid_uav[(valid_uav["mode"] == mode_key) & (valid_uav["feasible"]) & (~valid_uav["pareto"])]
    sub_no = valid_uav[(valid_uav["mode"] == mode_key) & (~valid_uav["feasible"])]
    ax.scatter(sub_par["T50"], sub_par["C"], color=COLORS[mode_key], s=195, marker="*", alpha=0.98, edgecolors="white", linewidth=0.7, label=f"{lbl} — множина Парето", zorder=5)
    ax.scatter(sub_ok["T50"], sub_ok["C"], color=COLORS[mode_key], s=64, alpha=0.92, edgecolors="white", linewidth=0.7, label=f"{lbl} — інші допустимі", zorder=4)
    ax.scatter(sub_no["T50"], sub_no["C"], facecolors="white", edgecolors=COLORS[mode_key], linewidth=1.3, s=52, marker="X", alpha=0.85, label=f"{lbl} — відсіяні", zorder=3)

label_pos = {
    "tev1-4b-experimental (txt)": (0.3265, 0.0134),
    "drex-max (MM)": (0.6614, 0.2964),
    "d1 (txt)": (0.3151, 0.5946),
    "d1 (MM)": (0.3651, 0.4703),
    "joey-1 (MM)": (0.3702, 0.6955),
    "kev-4b (txt)": (0.5287, 0.7714),
    "drex-v1.5 (txt)": (0.6215, 0.6592),
    "joey-1 (txt)": (0.2632, 0.6182),
    "microsoft-decision-1 (txt)": (0.5025, 0.8991),
    "clef-omni (MM)": (0.2760, 1.0982),
    "d1-ultra (txt)": (0.3299, 0.9917),
    "pplx-decider-v1.2 (MM)": (0.4294, 1.0883),
    "d1.1 (MM)": (0.3418, 1.2869),
    "span-01 (txt)": (0.4150, 1.5050),
    "classify-1 (txt)": (0.4880, 1.5650),
    "clef-omni (txt)": (0.2325, 1.4773),
    "pplx-decider-flash-v1.2 (txt)": (0.3615, 1.7029),
    "d1.1 (txt)": (0.3263, 1.8972),
    "clef-flash (txt)": (0.4825, 1.8869),
    "span-02 (txt)": (0.5565, 1.8875),
    "drex-v2.0 (txt)": (0.6627, 1.6957),
    "d1-ultra (MM)": (0.4366, 2.1210),
    "span-02 (MM)": (0.4136, 1.8653),
    "clef (txt)": (0.5842, 2.1072),
    "jev-1.13 (txt)": (0.2339, 2.0636),
    "gpt-6-luna-decisions (txt)": (0.3326, 2.0896),
    "pplx-decider-v1.2 (txt)": (0.4439, 2.3236),
    "drex-max (txt)": (0.6713, 2.0868),
    "jev-1.13 (MM)": (0.2542, 2.4884),
    "elephant-1 (MM)": (0.3627, 2.2154),
    "clef (MM)": (0.5161, 2.5118),
    "mammoth-1 (txt)": (0.3292, 2.3221),
    "gpt-6-luna-decisions (MM)": (0.3737, 2.5148),
    "jev-1.14 (txt)": (0.2430, 2.9195),
    "elephant-1 (txt)": (0.2780, 2.7129),
    "pplx-decider-flash-v1.2 (MM)": (0.3905, 2.9432),
    "clef-flash (MM)": (0.2437, 3.4608),
    "mammoth-1 (MM)": (0.4097, 3.2378),
    "jev-1.14 (MM)": (0.3145, 3.3070),
}
for _, r in valid_uav.iterrows():
    tx, ty = label_pos.get(r["label"], (r["T50"] + 0.015, r["C"] + 0.12))
    is_ok = bool(r["feasible"])
    ax.annotate(
        r["label"],
        xy=(r["T50"], r["C"]),
        xytext=(tx, ty),
        textcoords="data",
        ha="center",
        va="center",
        fontsize=7.3,
        color="black" if is_ok else "0.38",
        fontweight="bold" if is_ok else "normal",
        arrowprops=dict(arrowstyle="-", color="0.45" if is_ok else "0.65", lw=0.7, shrinkA=1, shrinkB=4),
        zorder=5 if is_ok else 4,
    )

ax.set_xlim(0.14, 0.76)
ax.set_ylim(-0.22, 3.72)
ax.xaxis.set_major_formatter(comma_axis(2))
ax.yaxis.set_major_formatter(comma_axis(1))
ax.set_xlabel("Медіанна затримка звернення до моделі $T_{50}$, с")
ax.set_ylabel("Середня кількість зіткнень $C$, од.")
ax.set_title("БпЛА: зіткнення $C$ та затримка $T_{50}$", pad=56)
ax.grid(True, alpha=0.3)
h_sc, l_sc = ax.get_legend_handles_labels()
ord_sc = [0, 3, 1, 4, 2, 5]
ax.legend([h_sc[i] for i in ord_sc], [l_sc[i] for i in ord_sc], loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=3, fontsize=8.4, frameon=True)
save_fig(fig, "fig4_3_uav_latency_collisions")

sub_lat_uav = uav_feasible.sort_values("T50", ascending=True).reset_index(drop=True)
yl_uav = np.arange(len(sub_lat_uav))

fig, ax = plt.subplots(figsize=(9.0, 6.6))
for i, r in sub_lat_uav.iterrows():
    col = COLORS[r["mode"]]
    ax.plot([r["T50"], r["T90"]], [i, i], color=col, lw=2.3, zorder=2)
    if r["pareto"]:
        ax.scatter(r["T50"], i, color=col, marker="*", s=140, zorder=4)
    else:
        ax.scatter(r["T50"], i, color=col, marker="o", s=52, zorder=3)
    ax.scatter(r["T90"], i, facecolors="white", edgecolors=col, linewidth=1.6, marker="s", s=50, zorder=4)
    ne_txt = "" if r["Ne"] == 0 else f", $N_e$ = {fmt_num(r['Ne'], 1)}"
    lbl_lat = f"[{fmt_num(r['T50'], 2)}; {fmt_num(r['T90'], 2)}] с ($n$ = {fmt_num(r['n'], 0)}{ne_txt}, $U$ = {fmt_num(r['U'] / 1000, 0)} тис.)"
    ax.text(r["T90"] + 0.02, i, lbl_lat, va="center", fontsize=8.1)

ax.scatter([], [], color="C0", marker="o", s=52, label="MM: медіана $T_{50}$")
ax.scatter([], [], facecolors="white", edgecolors="C0", linewidth=1.6, marker="s", s=50, label="MM: 90-й перцентиль $T_{90}$")
ax.scatter([], [], color="C1", marker="o", s=52, label="txt: медіана $T_{50}$")
ax.scatter([], [], facecolors="white", edgecolors="C1", linewidth=1.6, marker="s", s=50, label="txt: 90-й перцентиль $T_{90}$")
ax.scatter([], [], color="black", marker="*", s=110, label="Множина Парето")
ax.set_yticks(yl_uav, [ylbl(r) for _, r in sub_lat_uav.iterrows()])
ax.invert_yaxis()
ax.set_xlim(0.20, 1.72)
ax.xaxis.set_major_formatter(comma_axis(2))
ax.set_xlabel(r"Тривалість звернення до моделі $[T_{50};\, T_{90}]$, с")
ax.set_title(r"БпЛА: затримки $[T_{50};\, T_{90}]$", pad=58)
ax.grid(axis="x", alpha=0.3)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.015), ncol=3, fontsize=8.2, frameon=True)
save_fig(fig, "fig4_3b_uav_latency_intervals")

fig, ax = plt.subplots(figsize=(8.6, 6.8))
h_u = 0.38
ax.barh(yc - h_u / 2, sub_c["Ra"], h_u, color="C0", label="Тактичний контур SOM ($R_a$)")
ax.barh(yc + h_u / 2, sub_c["Rr"], h_u, color="C1", label="Рефлекс безпеки ($R_r$)")
ax.set_yticks(yc, [ylbl(r) for _, r in sub_c.iterrows()])
ax.invert_yaxis()
ax.set_xlim(0, 102)
ax.set_xticks([0, 20, 40, 60, 80, 100])
ax.xaxis.set_major_formatter(comma_axis(0))
ax.set_xlabel("Частка тактів навігації, %")
ax.set_title("БпЛА: активність контурів керування", pad=40)
ax.grid(axis="x", alpha=0.3)
ctrl_h = [patches.Patch(color="C0", label="Тактичний контур SOM ($R_a$)"), patches.Patch(color="C1", label="Рефлекс безпеки ($R_r$)"), Line2D([], [], marker="*", color="none", markerfacecolor="black", markeredgecolor="black", markersize=9, label="Множина Парето")]
ax.legend(handles=ctrl_h, loc="lower center", bbox_to_anchor=(0.5, 1.015), ncol=3, fontsize=8.3, frameon=True)
save_fig(fig, "fig4_3c_uav_control_ticks")


In [ ]:
STATIC_UAV_OBSTACLES = {
    "cylinders": [
        (7.0, 1.8, 2.5, 0.35, 2.5), (10.0, -1.8, 2.5, 0.35, 2.5), (13.0, 1.8, 2.5, 0.35, 2.5),
        (50.0, 1.5, 2.5, 0.35, 2.5), (51.0, -1.6, 2.5, 0.35, 2.5), (54.0, 0.2, 2.5, 0.35, 2.5),
        (57.0, -2.2, 2.5, 0.35, 2.5), (58.0, 2.0, 2.5, 0.35, 2.5),
    ],
    "beams": [(19.0, 0.0, 1.60, 0.50, 45.0), (44.0, 0.0, 1.60, 0.50, 45.0)],
    "gates": [(38.0, 31.6, 4.5, 0.25, 30.0, 4.5), (38.0, -31.6, 4.5, 0.25, 30.0, 4.5)],
}

uav_pareto_top2 = df_uav[df_uav["pareto"]].sort_values(["X", "C"], ascending=[False, True]).head(2)
run1_uav = df_runs[(df_runs["domain"] == "drone") & (df_runs["run"] == "run_1")].set_index("config")
uav_crashed_r1 = df_runs[(df_runs["domain"] == "drone") & (df_runs["run"] == "run_1") & (df_runs["S"] == 0.0) & (df_runs["X"] > 19.0)].sort_values("X", ascending=True).iloc[0]
uav_cases = [
    ("heuristic", "Базова евристика", "C7", 2.0, "o"),
    (uav_crashed_r1["config"], f"{uav_crashed_r1['label']} (відсіяна)", "C3", 2.0, "s"),
    (uav_pareto_top2.iloc[0]["config"], f"★ {uav_pareto_top2.iloc[0]['label']}", "C0", 2.5, "*"),
    (uav_pareto_top2.iloc[1]["config"], f"★ {uav_pareto_top2.iloc[1]['label']}", "C2", 1.8, "^"),
]

uav_traces = {}
for cfg, short_lbl, col, lw, mk in uav_cases:
    arr = np.array(run1_trajectories[("drone", cfg)], dtype=float)
    row1 = run1_uav.loc[cfg]
    lbl = f"{short_lbl} ($X$ = {fmt_num(row1['X'], 1)} м, $C$ = {int(row1['C'])})"
    uav_traces[cfg] = (arr[:, 0], arr[:, 1], arr[:, 2], arr[:, 3], lbl, col, lw, mk)

uav_obs = STATIC_UAV_OBSTACLES
fig, ax = plt.subplots(figsize=(12.2, 4.0))
ax.scatter([c[0] for c in uav_obs["cylinders"]], [c[1] for c in uav_obs["cylinders"]], s=58, facecolor="lightgray", edgecolor="dimgray", lw=0.9, label="Тверді колони", zorder=3)
for idx, (x, y, z, r, hl) in enumerate(uav_obs["beams"]):
    ax.add_patch(patches.Rectangle((x - r, -12.0), 2 * r, 20.5, facecolor="wheat", edgecolor="peru", lw=0.8, alpha=0.85, label="Низькі балки ($x = 19, 44$ м)" if idx == 0 else None, zorder=2))
for idx, (x, y, z, sx, sy, sz) in enumerate(uav_obs["gates"]):
    y0 = max(-12.0, y - sy)
    y1 = min(8.5, y + sy)
    if y1 > y0:
        ax.add_patch(patches.Rectangle((x - sx, y0), 2 * sx, y1 - y0, facecolor="#9bd3e6", edgecolor="#3b8ea5", lw=0.8, alpha=0.75, label="Рухомі ворота ($x = 38$ м, $t = 0$)" if idx == 0 else None, zorder=2))

for cfg, (_, x_a, y_a, _, lbl, col, lw, mk) in uav_traces.items():
    ax.plot(x_a, y_a, color=col, lw=lw, label=lbl, zorder=4)
    ax.scatter(x_a[-1], y_a[-1], color=col, marker=mk, s=55 if mk == "o" else 65, edgecolors="white", linewidth=0.7, zorder=6 if mk == "^" else 5)

ax.set_xlim(0, 120)
ax.set_ylim(-12.0, 8.5)
ax.set_yticks([-12, -8, -4, 0, 4, 8])
ax.xaxis.set_major_formatter(comma_axis(0))
ax.yaxis.set_major_formatter(comma_axis(0))
ax.set_xlabel("Поздовжня координата $x$, м")
ax.set_ylabel("Поперечна координата $y$, м")
ax.set_title("БпЛА: траєкторії польоту ($XY$)", pad=46)
ax.grid(True, alpha=0.3)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.03), ncol=4, fontsize=7.9, frameon=True)
save_fig(fig, "fig4_3d_uav_trajectories_xy")

fig, ax = plt.subplots(figsize=(12.2, 4.2))
for idx, (x, y, z, r, hz) in enumerate(uav_obs["cylinders"]):
    ax.add_patch(patches.Rectangle((x - r, 0), 2 * r, 2 * hz, facecolor="lightgray", edgecolor="dimgray", lw=0.6, alpha=0.35, label="Тверді колони (проєкція $XZ$)" if idx == 0 else None, zorder=1))
for idx, (x, y, z, r, hl) in enumerate(uav_obs["beams"]):
    ax.add_patch(patches.Rectangle((x - r, z - r), 2 * r, 2 * r, facecolor="wheat", edgecolor="peru", lw=1.0, label="Низькі балки ($x = 19, 44$ м)" if idx == 0 else None, zorder=3))
for idx, (x, y, z, sx, sy, sz) in enumerate(uav_obs["gates"]):
    if idx == 0:
        ax.add_patch(patches.Rectangle((x - sx, 0), 2 * sx, 2 * sz, facecolor="#9bd3e6", edgecolor="#3b8ea5", lw=0.7, alpha=0.35, label="Рухомі ворота ($x = 38$ м)" if idx == 0 else None, zorder=1))

for cfg, (_, x_a, _, z_a, lbl, col, lw, mk) in uav_traces.items():
    ax.plot(x_a, z_a, color=col, lw=lw, label=lbl, zorder=4)
    ax.scatter(x_a[-1], z_a[-1], color=col, marker=mk, s=55 if mk == "o" else 65, edgecolors="white", linewidth=0.7, zorder=6 if mk == "^" else 5)

ax.set_xlim(0, 120)
ax.set_ylim(0, 14.5)
ax.xaxis.set_major_formatter(comma_axis(0))
ax.yaxis.set_major_formatter(comma_axis(0))
ax.set_xlabel("Поздовжня координата $x$, м")
ax.set_ylabel("Висота польоту $z$, м")
ax.set_title("БпЛА: профіль висоти ($XZ$)", pad=46)
ax.grid(True, alpha=0.3)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.03), ncol=4, fontsize=7.9, frameon=True)
save_fig(fig, "fig4_3e_uav_trajectories_xz")


## 4. Основні графіки для Розділу 4: НРК та крос-доменне зведення (п. 4.3–4.4)
* **Рис. 4.4** — Просування НРК по трасі $X$ та зіткнення $C$ для допустимих конфігурацій НРК ($P_b = 100\,\%$) та «Базової евристики» (`★` — множина Парето).
* **Рис. 4.4б** — Середня кількість аварійних зіткнень $C$ допустимих конфігурацій НРК.
* **Рис. 4.4в** — Зв’язок медіанної затримки $T_{50}$ і зіткнень $C$ по всіх 39 валідних конфігураціях SOM НРК (зірочки — множина Парето, кола — інші допустимі, хрестики — відсіяні).
* **Рис. 4.4г** — Інтервали обчислювальної затримки $[T_{50};\, T_{90}]$, виклики $n$ і витрата токенів $U$ для допустимих конфігурацій НРК.
* **Рис. 4.4д** — Розподіл тактів керування НРК між моделлю SOM ($R_a$), локальною евристикою та вільним рухом.
* **Рис. 4.5** — Траєкторії прориву картонної стіни НРК у проєкції $XY$ (усі 3 моделі з множини Парето).
* **Рис. 4.6** — Крос-доменна ефективність конфігурацій, що пройшли жорсткий фільтр на обох платформах.

In [ ]:
yr = np.arange(len(ugv_sel))

fig, ax = plt.subplots(figsize=(8.8, 3.9))
ax.barh(yr, ugv_sel["X"], color=[COLORS[m] for m in ugv_sel["mode"]])
for i, r in ugv_sel.iterrows():
    lbl_txt = f"{fmt_num(r['X'], 1)} м ($C$ = {fmt_num(r['C'], 1)})"
    ax.text(r["X"] + 1.5, i, lbl_txt, va="center", fontsize=8.4)
ax.set_yticks(yr, [ylbl(r) for _, r in ugv_sel.iterrows()])
ax.invert_yaxis()
ax.set_xlim(0, 118)
ax.set_xticks([0, 20, 40, 60, 80, 100])
ax.xaxis.set_major_formatter(comma_axis(0))
ax.set_xlabel("Максимальне просування по трасі $X$, м")
ax.set_title("НРК: просування по трасі $X$", pad=54)
ax.grid(axis="x", alpha=0.3)
ax.legend(handles=mode_handles(True, True), loc="lower center", bbox_to_anchor=(0.5, 1.03), ncol=2, frameon=True)
save_fig(fig, "fig4_4_ugv_progress")

sub_c_ugv = pd.concat([ugv_feasible.sort_values(["C", "X"], ascending=[True, False]), df_ugv[df_ugv["mode"] == "heuristic"]]).reset_index(drop=True)
yc_ugv = np.arange(len(sub_c_ugv))

fig, ax = plt.subplots(figsize=(8.8, 3.9))
ax.barh(yc_ugv, sub_c_ugv["C"], color=[COLORS[m] for m in sub_c_ugv["mode"]])
for i, r in sub_c_ugv.iterrows():
    lbl_c = f"{fmt_num(r['C'], 1)} ({fmt_num(r['X'], 1)} м)"
    ax.text(r["C"] + 0.025, i, lbl_c, va="center", fontsize=8.4)
ax.set_yticks(yc_ugv, [ylbl(r) for _, r in sub_c_ugv.iterrows()])
ax.invert_yaxis()
ax.set_xlim(0, 1.48)
ax.set_xticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax.xaxis.set_major_formatter(comma_axis(1))
ax.set_xlabel("Середня кількість унікальних зіткнень $C$, од.")
ax.set_title("НРК: зіткнення $C$", pad=54)
ax.grid(axis="x", alpha=0.3)
ax.legend(handles=mode_handles(True, True), loc="lower center", bbox_to_anchor=(0.5, 1.03), ncol=2, frameon=True)
save_fig(fig, "fig4_4b_ugv_collisions")

valid_ugv = som_ugv[~som_ugv["low_lat_validity"]].copy()

fig, ax = plt.subplots(figsize=(13.4, 8.6))
for mode_key, lbl in [("multimodal", "MM"), ("text", "txt")]:
    sub_par = valid_ugv[(valid_ugv["mode"] == mode_key) & (valid_ugv["pareto"])]
    sub_ok = valid_ugv[(valid_ugv["mode"] == mode_key) & (valid_ugv["feasible"]) & (~valid_ugv["pareto"])]
    sub_no = valid_ugv[(valid_ugv["mode"] == mode_key) & (~valid_ugv["feasible"])]
    ax.scatter(sub_par["T50"], sub_par["C"], color=COLORS[mode_key], s=195, marker="*", alpha=0.98, edgecolors="white", linewidth=0.7, label=f"{lbl} — множина Парето", zorder=5)
    ax.scatter(sub_ok["T50"], sub_ok["C"], color=COLORS[mode_key], s=64, alpha=0.92, edgecolors="white", linewidth=0.7, label=f"{lbl} — інші допустимі", zorder=4)
    ax.scatter(sub_no["T50"], sub_no["C"], facecolors="white", edgecolors=COLORS[mode_key], linewidth=1.3, s=52, marker="X", alpha=0.85, label=f"{lbl} — відсіяні", zorder=3)

ugv_label_pos = {
    "jev-1.14 (MM)": (0.2678, -0.0626),
    "elephant-1 (txt)": (0.2699, -0.1196),
    "d1.1 (txt)": (0.2759, 0.0579),
    "joey-1 (txt)": (0.3190, 0.1134),
    "d1 (txt)": (0.3780, -0.0880),
    "span-02 (txt)": (0.3311, 0.0521),
    "elephant-1 (MM)": (0.3520, -0.1580),
    "gpt-6-luna-decisions (MM)": (0.2829, -0.1950),
    "d1 (MM)": (0.3642, 0.1082),
    "d1-ultra (MM)": (0.4480, 0.0386),
    "span-01 (txt)": (0.4400, -0.0880),
    "microsoft-decision-1 (txt)": (0.5334, 0.0279),
    "span-02 (MM)": (0.7015, -0.0420),
    "clef-omni (txt)": (0.2473, 0.3121),
    "jev-1.13 (txt)": (0.2495, 0.1359),
    "gpt-6-luna-decisions (txt)": (0.2308, 0.2365),
    "clef-omni (MM)": (0.3462, 0.2186),
    "pplx-decider-v1.2 (txt)": (0.4964, 0.1205),
    "jev-1.13 (MM)": (0.2530, 0.4487),
    "d1.1 (MM)": (0.3281, 0.4582),
    "joey-1 (MM)": (0.3817, 0.4244),
    "clef (MM)": (0.5361, 0.3545),
    "classify-1 (txt)": (0.6760, 0.4839),
    "drex-max (txt)": (0.7440, 0.4483),
    "drex-max (MM)": (0.7092, 0.3511),
    "clef-flash (MM)": (0.2404, 0.6127),
    "d1-ultra (txt)": (0.3702, 0.6597),
    "jev-1.14 (txt)": (0.2622, 0.7424),
    "mammoth-1 (txt)": (0.2889, 0.8429),
    "mammoth-1 (MM)": (0.3780, 0.7364),
    "tev1-4b-experimental (txt)": (0.3673, 0.8981),
    "drex-v1.5 (txt)": (0.5678, 0.8260),
    "pplx-decider-flash-v1.2 (txt)": (0.2560, 1.0635),
    "pplx-decider-flash-v1.2 (MM)": (0.3120, 1.1350),
    "pplx-decider-v1.2 (MM)": (0.4566, 1.0403),
    "clef-flash (txt)": (0.4823, 1.1019),
    "kev-4b (txt)": (0.5870, 0.9041),
    "clef (txt)": (0.5601, 1.1514),
    "drex-v2.0 (txt)": (0.6293, 1.4578),
}
for _, r in valid_ugv.iterrows():
    tx, ty = ugv_label_pos.get(r["label"], (r["T50"] + 0.02, r["C"] + 0.08))
    is_ok = bool(r["feasible"])
    ax.annotate(
        r["label"],
        xy=(r["T50"], r["C"]),
        xytext=(tx, ty),
        textcoords="data",
        ha="center",
        va="center",
        fontsize=7.3,
        color="black" if is_ok else "0.38",
        fontweight="bold" if is_ok else "normal",
        arrowprops=dict(arrowstyle="-", color="0.45" if is_ok else "0.65", lw=0.7, shrinkA=1, shrinkB=4),
        zorder=5 if is_ok else 4,
    )

ax.set_xlim(0.14, 0.82)
ax.set_ylim(-0.28, 1.56)
ax.set_yticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4])
ax.xaxis.set_major_formatter(comma_axis(2))
ax.yaxis.set_major_formatter(comma_axis(1))
ax.set_xlabel("Медіанна затримка звернення до моделі $T_{50}$, с")
ax.set_ylabel("Середня кількість зіткнень $C$, од.")
ax.set_title("НРК: зіткнення $C$ та затримка $T_{50}$", pad=56)
ax.grid(True, alpha=0.3)
h_sc, l_sc = ax.get_legend_handles_labels()
ord_sc = [0, 3, 1, 4, 2, 5]
ax.legend([h_sc[i] for i in ord_sc], [l_sc[i] for i in ord_sc], loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=3, fontsize=8.4, frameon=True)
save_fig(fig, "fig4_4c_ugv_latency_collisions")

sub_lat_ugv = ugv_feasible.sort_values("T50", ascending=True).reset_index(drop=True)
yl_ugv = np.arange(len(sub_lat_ugv))

fig, ax = plt.subplots(figsize=(9.0, 3.8))
for i, r in sub_lat_ugv.iterrows():
    col = COLORS[r["mode"]]
    ax.plot([r["T50"], r["T90"]], [i, i], color=col, lw=2.3, zorder=2)
    if r["pareto"]:
        ax.scatter(r["T50"], i, color=col, marker="*", s=140, zorder=4)
    else:
        ax.scatter(r["T50"], i, color=col, marker="o", s=52, zorder=3)
    ax.scatter(r["T90"], i, facecolors="white", edgecolors=col, linewidth=1.6, marker="s", s=50, zorder=4)
    ne_txt = "" if r["Ne"] == 0 else f", $N_e$ = {fmt_num(r['Ne'], 1)}"
    lbl_lat = f"[{fmt_num(r['T50'], 2)}; {fmt_num(r['T90'], 2)}] с ($n$ = {fmt_num(r['n'], 0)}{ne_txt}, $U$ = {fmt_num(r['U'] / 1000, 0)} тис.)"
    ax.text(r["T90"] + 0.025, i, lbl_lat, va="center", fontsize=8.2)

ax.scatter([], [], color="C0", marker="o", s=52, label="MM: медіана $T_{50}$")
ax.scatter([], [], facecolors="white", edgecolors="C0", linewidth=1.6, marker="s", s=50, label="MM: 90-й перцентиль $T_{90}$")
ax.scatter([], [], color="C1", marker="o", s=52, label="txt: медіана $T_{50}$")
ax.scatter([], [], facecolors="white", edgecolors="C1", linewidth=1.6, marker="s", s=50, label="txt: 90-й перцентиль $T_{90}$")
ax.scatter([], [], color="black", marker="*", s=110, label="Множина Парето")
ax.set_yticks(yl_ugv, [ylbl(r) for _, r in sub_lat_ugv.iterrows()])
ax.invert_yaxis()
ax.set_xlim(0.20, 1.92)
ax.xaxis.set_major_formatter(comma_axis(2))
ax.set_xlabel(r"Тривалість звернення до моделі $[T_{50};\, T_{90}]$, с")
ax.set_title(r"НРК: затримки $[T_{50};\, T_{90}]$", pad=58)
ax.grid(axis="x", alpha=0.3)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.03), ncol=3, fontsize=8.2, frameon=True)
save_fig(fig, "fig4_4d_ugv_latency_intervals")

fig, ax = plt.subplots(figsize=(8.6, 3.9))
ax.barh(yr, ugv_sel["Ra"], color="C0")
ax.barh(yr, ugv_sel["R_heu"], left=ugv_sel["Ra"], color="C1")
ax.barh(yr, ugv_sel["R_clr"], left=ugv_sel["Ra"] + ugv_sel["R_heu"], color="C7")
ax.set_yticks(yr, [ylbl(r) for _, r in ugv_sel.iterrows()])
ax.invert_yaxis()
ax.set_xlim(0, 102)
ax.set_xticks([0, 20, 40, 60, 80, 100])
ax.xaxis.set_major_formatter(comma_axis(0))
ax.set_xlabel("Частка тактів керування за контурами, %")
ax.set_title("НРК: розподіл тактів керування", pad=54)
ax.grid(axis="x", alpha=0.3)
ugv_ctrl_h = [
    patches.Patch(color="C0", label="Модель SOM ($R_a$)"),
    patches.Patch(color="C1", label=r"Локальна евристика ($R_{\mathrm{евр}}$)"),
    patches.Patch(color="C7", label=r"Вільний рух ($R_{\mathrm{віл}}$)"),
    Line2D([], [], marker="*", color="none", markerfacecolor="black", markeredgecolor="black", markersize=9, label="Множина Парето"),
]
ax.legend(handles=ugv_ctrl_h, loc="lower center", bbox_to_anchor=(0.5, 1.03), ncol=2, fontsize=8.4, frameon=True)
save_fig(fig, "fig4_4e_ugv_control_ticks")


In [ ]:
STATIC_OBSTACLES = {
    "cylinders": [
        (7.0, -2.5, 0.35), (7.0, 5.0, 0.35), (11.5, -6.2, 0.35), (11.5, 1.2, 0.35),
        (16.0, -2.5, 0.35), (16.0, 5.0, 0.35), (20.5, -6.2, 0.35), (20.5, 1.2, 0.35),
        (51.7, 0.4, 0.35), (59.0, 3.6, 0.35), (54.4, 4.2, 0.35), (62.6, 6.2, 0.35),
        (63.5, -3.3, 0.35), (65.0, 0.8, 0.35),
    ],
    "overpass_sups": [(28.5, -5.0, 3.0, 0.5), (28.5, 5.0, 3.0, 0.5)],
    "boxes": [(28.0, round(-6.6 + 1.1 * k, 1), 0.5, 0.5) for k in range(13)],
    "gates": [(38.0, 31.6, 0.25, 30.0), (38.0, -31.6, 0.25, 30.0)],
}

ugv_pareto_all = df_ugv[df_ugv["pareto"]].sort_values("X", ascending=False)
ugv_paralyzed = som_ugv[~som_ugv["feasible"]].sort_values("X", ascending=True).iloc[0]
ugv_cases = [
    ("heuristic", "Базова евристика", "C7", 2.0, "o"),
    (ugv_paralyzed["config"], f"{ugv_paralyzed['label']} (відсіяна)", "C3", 2.0, "s"),
]
for (_, r), col, lw, mk in zip(ugv_pareto_all.iterrows(), ["C0", "C2", "C1", "C4"], [2.4, 1.8, 1.6, 1.5], ["*", "^", "D", "v"]):
    ugv_cases.append((r["config"], f"★ {r['label']}", col, lw, mk))

run1_ugv = df_runs[(df_runs["domain"] == "rover") & (df_runs["run"] == "run_1")].set_index("config")
traces = {}
for cfg, short_lbl, col, lw, mk in ugv_cases:
    traj = run1_trajectories[("rover", cfg)]
    t_a = np.array([pt[0] for pt in traj], dtype=float)
    x_a = np.array([pt[1] for pt in traj], dtype=float)
    y_a = np.array([pt[2] for pt in traj], dtype=float)
    ov_p = np.array([np.nan if pt[5] is None else float(pt[5]) for pt in traj], dtype=float)
    row1 = run1_ugv.loc[cfg]
    lbl = f"{short_lbl} ($X$ = {fmt_num(row1['X'], 1)} м, $C$ = {int(row1['C'])})"
    traces[cfg] = (t_a, x_a, y_a, ov_p, lbl, short_lbl, col, lw, mk)

obs = STATIC_OBSTACLES
fig, ax = plt.subplots(figsize=(12.2, 3.6))

for idx, (x, y, r) in enumerate(obs["cylinders"]):
    ax.add_patch(patches.Circle((x, y), r, facecolor="lightgray", edgecolor="dimgray", lw=0.8, label="Тверді колони" if idx == 0 else None, zorder=2))

for idx, (x, y, sx, sy) in enumerate(obs["overpass_sups"]):
    ax.add_patch(patches.Rectangle((x - sx, y - sy), 2 * sx, 2 * sy, facecolor="silver", edgecolor="dimgray", lw=0.8, label="Опори естакади" if idx == 0 else None, zorder=2))

for idx, (x, y, sx, sy) in enumerate(obs["boxes"]):
    ax.add_patch(patches.Rectangle((x - sx, y - sy), 2 * sx, 2 * sy, facecolor="wheat", edgecolor="peru", lw=0.8, label="Картонна стіна ($x = 28$ м)" if idx == 0 else None, zorder=2))

for idx, (x, y, sx, sy) in enumerate(obs["gates"]):
    y0 = max(-6.5, y - sy)
    y1 = min(6.5, y + sy)
    if y1 > y0:
        ax.add_patch(patches.Rectangle((x - sx, y0), 2 * sx, y1 - y0, facecolor="#9bd3e6", edgecolor="#3b8ea5", lw=0.8, alpha=0.75, label="Рухомі ворота ($x = 38$ м, $t = 0$)" if idx == 0 else None, zorder=2))

for cfg, (t_a, x_a, y_a, _, lbl, _, col, lw, mk) in traces.items():
    ax.plot(x_a, y_a, color=col, lw=lw, label=lbl, zorder=4)
    ax.scatter(x_a[-1], y_a[-1], color=col, marker=mk, s=55 if mk == "o" else 65, edgecolors="white", linewidth=0.7, zorder=6 if mk == "^" else 5)

ax.set_aspect("equal")
ax.set_xlim(-1, 90)
ax.set_ylim(-6.5, 6.5)
ax.xaxis.set_major_formatter(comma_axis(0))
ax.yaxis.set_major_formatter(comma_axis(0))
ax.set_xlabel("Поздовжня координата $x$, м")
ax.set_ylabel("Поперечна координата $y$, м")
ax.set_title("НРК: траєкторії руху ($XY$)", pad=64)
ax.grid(True, alpha=0.3)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.04), ncol=4, fontsize=7.8, frameon=True)
save_fig(fig, "fig4_5_ugv_trajectories")

fig, ax = plt.subplots(figsize=(8.8, 3.8))
yk = np.arange(len(cross_sel))
h = 0.38
uav_cols = ["C7" if m == "heuristic" else "C0" for m in cross_sel["mode"]]
ugv_cols = ["silver" if m == "heuristic" else "C1" for m in cross_sel["mode"]]
ax.barh(yk - h / 2, cross_sel["uav_pct"], h, color=uav_cols)
ax.barh(yk + h / 2, cross_sel["ugv_pct"], h, color=ugv_cols)
for i, r in cross_sel.iterrows():
    if bool(r["pareto_uav"]):
        ax.scatter(r["uav_pct"] - 3.6, i - h / 2, marker="*", s=130, facecolor="white", edgecolor="black", linewidth=0.7, zorder=5)
    if bool(r["pareto_ugv"]):
        ax.scatter(r["ugv_pct"] - 3.6, i + h / 2, marker="*", s=130, facecolor="white", edgecolor="black", linewidth=0.7, zorder=5)
    ax.text(max(r["uav_pct"], r["ugv_pct"]) + 1.5, i, f"{r['uav_pct']:.0f} % / {r['ugv_pct']:.0f} %", va="center", fontsize=8.5)
ax.set_yticks(yk, cross_sel["label"])
ax.invert_yaxis()
ax.set_xlim(0, 124)
ax.set_xticks([0, 20, 40, 60, 80, 100])
ax.xaxis.set_major_formatter(comma_axis(0))
ax.set_xlabel("Частка проходження маршруту, %")
ax.set_title("Крос-доменне проходження маршруту", pad=76)
ax.grid(axis="x", alpha=0.3)
cd_handles = [
    patches.Patch(color="C0", label=f"БпЛА — SOM (% від {fmt_num(uav_max_x, 1)} м)"),
    patches.Patch(color="C1", label=f"НРК — SOM (% від {fmt_num(ugv_max_x, 1)} м)"),
    patches.Patch(color="C7", label="БпЛА — Базова евристика"),
    patches.Patch(color="silver", label="НРК — Базова евристика"),
    Line2D([], [], marker="*", color="none", markerfacecolor="white", markeredgecolor="black", markersize=9, label="Множина Парето платформи"),
]
ax.legend(handles=cd_handles, loc="lower center", bbox_to_anchor=(0.5, 1.03), ncol=2, frameon=True)
save_fig(fig, "fig4_6_cross_domain")


## 5. Додаткові графіки (відсіяні моделі, профіль override та порівняння режимів)
* **Рис. А.1** — Відсіяні жорстким фільтром конфігурації БпЛА із зазначенням причини відхилення ($S < 100\,\%$, $P_b < 100\,\%$ або $N_e > 1$).
* **Рис. А.1б** — Відсіяні жорстким фільтром конфігурації НРК із зазначенням причини відхилення ($P_b < 100\,\%$ або $N_e > 1$).
* **Рис. А.2** — Профіль `override_safety_reflex` за координатою $x$ для моделей із множини Парето НРК.
* **Рис. А.3** — Парне порівняння текстового та мультимодального режимів на БпЛА.
* **Рис. А.4** — Парне порівняння текстового та мультимодального режимів на НРК.

In [ ]:
uav_rejected = som_uav[~som_uav["feasible"]].sort_values("X", ascending=False).head(12).reset_index(drop=True)
ya = np.arange(len(uav_rejected))

fig, ax = plt.subplots(figsize=(9.2, 5.4))
ax.barh(ya, uav_rejected["X"], color=[COLORS[m] for m in uav_rejected["mode"]])
for i, r in uav_rejected.iterrows():
    reasons = []
    if r["S"] < 100.0:
        reasons.append(f"$S$ = {r['S']:.0f} %, $T$ = {fmt_num(r['T'], 1)} с")
    if r["Pb"] < 100.0:
        reasons.append(f"$P_b$ = {r['Pb']:.0f} %")
    if r["Ne"] > 1.0:
        reasons.append(f"$N_e$ = {fmt_num(r['Ne'], 1)}")
    lbl_ext = f"{fmt_num(r['X'], 1)} м (" + ", ".join(reasons) + ")"
    ax.text(r["X"] + 1.5, i, lbl_ext, va="center", fontsize=8.2)
ax.set_yticks(ya, uav_rejected["label"])
ax.invert_yaxis()
ax.set_xlim(0, 198)
ax.set_xticks([0, 20, 40, 60, 80, 100, 120])
ax.xaxis.set_major_formatter(comma_axis(0))
ax.set_xlabel("Максимальне просування по трасі $X$, м")
ax.set_title("БпЛА: відсіяні моделі", pad=34)
ax.grid(axis="x", alpha=0.3)
ax.legend(handles=mode_handles(False), loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=2, frameon=True)
save_fig(fig, "figA_1_uav_rejected")

ugv_rejected = som_ugv[~som_ugv["feasible"]].sort_values("X", ascending=False).head(12).reset_index(drop=True)
yr_rej = np.arange(len(ugv_rejected))

fig, ax = plt.subplots(figsize=(9.2, 5.4))
ax.barh(yr_rej, ugv_rejected["X"], color=[COLORS[m] for m in ugv_rejected["mode"]])
for i, r in ugv_rejected.iterrows():
    reasons = []
    if r["Pb"] < 100.0:
        reasons.append(f"$P_b$ = {r['Pb']:.0f} %")
    if r["Ne"] > 1.0:
        reasons.append(f"$N_e$ = {fmt_num(r['Ne'], 1)}")
    lbl_ext = f"{fmt_num(r['X'], 1)} м (" + ", ".join(reasons) + ")"
    ax.text(r["X"] + 1.2, i, lbl_ext, va="center", fontsize=8.2)
ax.set_yticks(yr_rej, ugv_rejected["label"])
ax.invert_yaxis()
ax.set_xlim(0, 132)
ax.set_xticks([0, 20, 40, 60, 80])
ax.xaxis.set_major_formatter(comma_axis(0))
ax.set_xlabel("Максимальне просування по трасі $X$, м")
ax.set_title("НРК: відсіяні моделі", pad=34)
ax.grid(axis="x", alpha=0.3)
ax.legend(handles=mode_handles(False), loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=2, frameon=True)
save_fig(fig, "figA_1b_ugv_rejected")

fig, ax = plt.subplots(figsize=(9.5, 4.1))
ugv_pareto_all = df_ugv[df_ugv["pareto"]].sort_values("X", ascending=False)
ugv_wall_stop = som_ugv[(~som_ugv["feasible"]) & (som_ugv["Pb"] == 0.0) & (som_ugv["Ne"] <= 1.0) & (som_ugv["X"] > 24.0)].sort_values("X", ascending=False).iloc[0]
ov_cases = [(r["config"], f"★ {r['label']}", col, lw) for (_, r), col, lw in zip(ugv_pareto_all.iterrows(), ["C0", "C2", "C1", "C4"], [2.4, 1.8, 1.6, 1.4])]
ov_cases.append((ugv_wall_stop["config"], f"{ugv_wall_stop['label']} (відсіяна)", "C3", 1.8))
for cfg, short_lbl, col, lw in ov_cases:
    traj = run1_trajectories[("rover", cfg)]
    x_a = np.array([pt[1] for pt in traj], dtype=float)
    ov_p = np.array([np.nan if pt[5] is None else float(pt[5]) for pt in traj], dtype=float)
    m = ~np.isnan(ov_p)
    ax.plot(x_a[m], ov_p[m], color=col, lw=lw, label=short_lbl)
ax.annotate("Картонна стіна\n($x = 28$ м)", xy=(28.0, 0.85), xytext=(39.0, 0.84),
            fontsize=9, ha="left", va="center",
            arrowprops=dict(arrowstyle="->", color="0.4", lw=1.0))
ax.set_xlim(0, 88)
ax.set_ylim(-0.05, 1.05)
ax.xaxis.set_major_formatter(comma_axis(0))
ax.yaxis.set_major_formatter(comma_axis(1))
ax.set_xlabel("Поздовжня координата $x$, м")
ax.set_ylabel("Коефіцієнт корекції рефлексу безпеки")
ax.set_title("НРК: придушення рефлексу безпеки", pad=48)
ax.grid(True, alpha=0.3)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.03), ncol=2, fontsize=8.4, frameon=True)
save_fig(fig, "figA_2_ugv_override")


In [ ]:
for df_p, df_dom, title, fname, xmax in [
    (df_pair_uav, df_uav, "БпЛА: текстовий vs мультимодальний режим", "figA_3_uav_multimodal", 142),
    (df_pair_ugv, df_ugv, "НРК: текстовий vs мультимодальний режим", "figA_4_ugv_multimodal", 110),
]:
    pareto_cfgs = set(df_dom[df_dom["pareto"]]["config"])
    work = df_p.copy()
    work["X_best"] = work[["X_tx", "X_mm"]].max(axis=1)
    sub_p = work.nlargest(10, "X_best").sort_values("X_best", ascending=False).reset_index(drop=True)
    sub_p["dx"] = sub_p["X_mm"] - sub_p["X_tx"]
    sub_p["p_tx"] = sub_p["config_tx"].isin(pareto_cfgs)
    sub_p["p_mm"] = sub_p["config_mm"].isin(pareto_cfgs)
    yp = np.arange(len(sub_p))

    fig, ax = plt.subplots(figsize=(8.6, 4.9))
    for i, r in sub_p.iterrows():
        ax.plot([r["X_tx"], r["X_mm"]], [i, i], color="C7", lw=2, zorder=2)
        ax.text(max(r["X_tx"], r["X_mm"]) + 2.4, i, f"{np.round(r['dx'], 1):+.1f} м".replace(".", ","), va="center", fontsize=9)
        if r["p_tx"]:
            ax.scatter(r["X_tx"], i, color="C1", marker="*", s=175, edgecolors="white", linewidth=0.6, zorder=5)
        else:
            ax.scatter(r["X_tx"], i, color="C1", marker="s", s=72, zorder=3)
        if r["p_mm"]:
            ax.scatter(r["X_mm"], i, color="C0", marker="*", s=175, edgecolors="white", linewidth=0.6, zorder=6)
        else:
            ax.scatter(r["X_mm"], i, facecolors="C0", edgecolors="white", linewidth=0.8, marker="o", s=46, zorder=4)

    p_ylbls = [f"★ {r['name_tx']}" if (r["p_tx"] or r["p_mm"]) else r["name_tx"] for _, r in sub_p.iterrows()]
    ax.set_yticks(yp, p_ylbls)
    ax.invert_yaxis()
    ax.set_xlim(0, xmax)
    ax.xaxis.set_major_formatter(comma_axis(0))
    ax.set_xlabel("Середнє просування по трасі $X$, м")
    ax.set_title(title, pad=40)
    ax.grid(axis="x", alpha=0.3)
    pair_h = [
        Line2D([], [], marker="s", color="none", markerfacecolor="C1", markeredgecolor="C1", markersize=7, label="Текстовий (txt)"),
        Line2D([], [], marker="o", color="none", markerfacecolor="C0", markeredgecolor="white", markersize=7, label="Мультимодальний (MM)"),
        Line2D([], [], marker="*", color="none", markerfacecolor="black", markeredgecolor="black", markersize=9, label="Множина Парето"),
    ]
    ax.legend(handles=pair_h, loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=3, fontsize=8.4, frameon=True)
    save_fig(fig, fname)


## 6. Компактні таблиці для статті, рангові кореляції Спірмена та повні таблиці Додатків

In [ ]:
t4_work = cross_sel.copy()
t4_work["U_uav_k"] = np.where(t4_work["mode"] == "heuristic", np.nan, t4_work["U_uav"] / 1000.0)
t4_work["U_ugv_k"] = np.where(t4_work["mode"] == "heuristic", np.nan, t4_work["U_ugv"] / 1000.0)

t4_main = t4_work[[
    "label",
    "X_uav", "Pb_uav", "V_uav", "D_uav", "C_uav", "S_uav", "T_uav",
    "Ra_uav", "Rr_uav", "T50_uav", "T90_uav", "n_uav", "Ne_uav", "U_uav_k",
    "X_ugv", "Pb_ugv", "C_ugv", "T_ugv",
    "Ra_ugv", "R_heu_ugv", "R_clr_ugv", "T50_ugv", "T90_ugv", "n_ugv", "Ne_ugv", "U_ugv_k",
]].round(2).copy()
t4_main.columns = [
    "Конфігурація",
    "X (БпЛА), м", "Pb (БпЛА), %", "V (БпЛА), %", "D (БпЛА), м",
    "C (БпЛА), од.", "S (БпЛА), %", "T (БпЛА), с",
    "Ra (БпЛА), %", "Rr (БпЛА), %",
    "T50 (БпЛА), с", "T90 (БпЛА), с", "n (БпЛА), од.", "Ne (БпЛА), од.", "U (БпЛА), тис.",
    "X (НРК), м", "Pb (НРК), %", "C (НРК), од.", "T (НРК), с",
    "Ra (НРК), %", "R_евр (НРК), %", "R_віл (НРК), %",
    "T50 (НРК), с", "T90 (НРК), с", "n (НРК), од.", "Ne (НРК), од.", "U (НРК), тис.",
]
t4_main.to_csv(os.path.join(TAB_DIR, "table4_top_models_compact.csv"), index=False, na_rep="—")

t5_rows = []
for dom_lbl, df_d, df_p in [("БпЛА", df_uav, df_pair_uav), ("НРК", df_ugv, df_pair_ugv)]:
    som_d = df_d[(df_d["mode"] != "heuristic") & (~df_d["low_lat_validity"])]
    h_d = df_d[df_d["mode"] == "heuristic"].iloc[0]
    t5_rows.append((
        dom_lbl, "Базова евристика", 1,
        h_d["X"], h_d["Pb"], h_d["V"], h_d["D"],
        h_d["C"], h_d["S"], h_d["T"],
        h_d["Ra"], h_d["Rr"], h_d["R_heu"], h_d["R_clr"],
        np.nan, np.nan, h_d["n"], h_d["Ne"], np.nan,
    ))
    for mode_lbl, sfx in [(f"Текстовий (парні {len(df_p)})", "tx"), (f"Мультимодальний (парні {len(df_p)})", "mm")]:
        t5_rows.append((
            dom_lbl, mode_lbl, len(df_p),
            df_p[f"X_{sfx}"].mean(), df_p[f"Pb_{sfx}"].mean(), df_p[f"V_{sfx}"].mean(), df_p[f"D_{sfx}"].mean(),
            df_p[f"C_{sfx}"].mean(), df_p[f"S_{sfx}"].mean(), df_p[f"T_{sfx}"].mean(),
            df_p[f"Ra_{sfx}"].mean(), df_p[f"Rr_{sfx}"].mean(), df_p[f"R_heu_{sfx}"].mean(), df_p[f"R_clr_{sfx}"].mean(),
            df_p[f"T50_{sfx}"].mean(), df_p[f"T90_{sfx}"].mean(),
            df_p[f"n_{sfx}"].mean(), df_p[f"Ne_{sfx}"].mean(), df_p[f"U_{sfx}"].mean() / 1000.0,
        ))
    for name, lo, hi in [("Швидкі (< 0,35 с)", 0.0, 0.35), ("Середні (0,35–0,55 с)", 0.35, 0.55), ("Повільні (> 0,55 с)", 0.55, 99.0)]:
        sub = som_d[(som_d["T50"] >= lo) & (som_d["T50"] < hi)]
        t5_rows.append((
            dom_lbl, name, len(sub),
            sub["X"].mean(), sub["Pb"].mean(), sub["V"].mean(), sub["D"].mean(),
            sub["C"].mean(), sub["S"].mean(), sub["T"].mean(),
            sub["Ra"].mean(), sub["Rr"].mean(), sub["R_heu"].mean(), sub["R_clr"].mean(),
            sub["T50"].mean(), sub["T90"].mean(),
            sub["n"].mean(), sub["Ne"].mean(), sub["U"].mean() / 1000.0,
        ))

t5_classes = pd.DataFrame(
    t5_rows,
    columns=[
        "Платформа", "Клас / режим", "Конфіг.",
        "X, м", "Pb, %", "V, %", "D, м",
        "C, од.", "S, %", "T, с",
        "Ra, %", "Rr, %", "R_евр, %", "R_віл, %",
        "T50, с", "T90, с", "n, од.", "Ne, од.", "U, тис.",
    ],
).round(2)
t5_classes.to_csv(os.path.join(TAB_DIR, "table5_classes_compact.csv"), index=False, na_rep="—")

p_uav = df_uav[df_uav["pareto"]][["label", "X", "V", "D", "C", "T50", "T90", "Ne", "U"]].copy()
p_uav.insert(0, "Платформа", "БпЛА")
p_uav["U"] = p_uav["U"] / 1000.0
p_ugv = df_ugv[df_ugv["pareto"]][["label", "X", "V", "D", "C", "T50", "T90", "Ne", "U"]].copy()
p_ugv.insert(0, "Платформа", "НРК")
p_ugv["U"] = p_ugv["U"] / 1000.0
t_pareto = pd.concat([p_uav, p_ugv], ignore_index=True).round(2)
t_pareto.columns = ["Платформа", "Конфігурація", "X, м", "V, %", "D, м", "C, од.", "T50, с", "T90, с", "Ne, од.", "U, тис."]
t_pareto.to_csv(os.path.join(TAB_DIR, "table_pareto_leaders.csv"), index=False, na_rep="—")

app_uav_cols = [
    "label", "provider", "mode", "feasible", "pareto", "X", "X_sd", "Pb", "V", "V_sd", "D", "D_sd",
    "C", "C_sd", "S", "T", "T_sd", "Ra", "Ra_sd", "Rr", "Rr_sd",
    "T50", "T50_sd", "T90", "T90_sd", "n", "n_sd", "Ne", "Ne_sd", "Ne_total",
    "U", "U_sd", "valid_lat_runs", "low_lat_validity",
]
app_ugv_cols = [
    "label", "provider", "mode", "feasible", "pareto", "X", "X_sd", "Pb", "C", "C_sd",
    "T", "T_sd", "Ra", "Ra_sd", "R_heu", "R_heu_sd", "R_clr", "R_clr_sd",
    "T50", "T50_sd", "T90", "T90_sd", "n", "n_sd", "Ne", "Ne_sd", "Ne_total",
    "U", "U_sd", "valid_lat_runs", "low_lat_validity",
]

app_uav = df_uav[app_uav_cols].round(2)
app_ugv = df_ugv[app_ugv_cols].round(2)
app_uav.to_csv(os.path.join(TAB_DIR, "appendix_A_uav_full_41.csv"), index=False, na_rep="—")
app_ugv.to_csv(os.path.join(TAB_DIR, "appendix_B_ugv_full_41.csv"), index=False, na_rep="—")

def spearman_rho(df_sub, col_x, col_y):
    sub = df_sub[[col_x, col_y]].dropna()
    if len(sub) < 3 or sub[col_x].nunique() < 2 or sub[col_y].nunique() < 2:
        return np.nan
    return float(sub[col_x].rank(method="average").corr(sub[col_y].rank(method="average")))

spearman_pairs_uav = [
    ("T50", "C", "T50 vs C"),
    ("T50", "X", "T50 vs X"),
    ("T50", "V", "T50 vs V"),
    ("T90", "C", "T90 vs C"),
    ("T90", "X", "T90 vs X"),
    ("Ra", "C", "Ra vs C"),
    ("Ra", "X", "Ra vs X"),
    ("Ra", "V", "Ra vs V"),
    ("Ra", "Rr", "Ra vs Rr"),
    ("U", "X", "U vs X"),
    ("U", "C", "U vs C"),
]
spearman_pairs_ugv = [
    ("T50", "C", "T50 vs C"),
    ("T50", "X", "T50 vs X"),
    ("T50", "Pb", "T50 vs Pb"),
    ("T90", "C", "T90 vs C"),
    ("T90", "X", "T90 vs X"),
    ("Ra", "C", "Ra vs C"),
    ("Ra", "X", "Ra vs X"),
    ("Ra", "Pb", "Ra vs Pb"),
    ("U", "X", "U vs X"),
    ("U", "C", "U vs C"),
]

spearman_pairs_ugv_feas = [p for p in spearman_pairs_ugv if p[1] != "Pb"]

sp_rows = []
for dom_lbl, sub_lbl, df_sub, pairs in [
    ("БпЛА", "Усі валідні SOM", valid_uav, spearman_pairs_uav),
    ("БпЛА", "Допустимі SOM", uav_feasible, spearman_pairs_uav),
    ("НРК", "Усі валідні SOM", valid_ugv, spearman_pairs_ugv),
    ("НРК", "Допустимі SOM", ugv_feasible, spearman_pairs_ugv_feas),
]:
    for cx, cy, pair_lbl in pairs:
        rho = spearman_rho(df_sub, cx, cy)
        sp_rows.append((dom_lbl, sub_lbl, len(df_sub), pair_lbl, rho))

t_spearman = pd.DataFrame(
    sp_rows,
    columns=["Платформа", "Вибірка", "k", "Пара метрик", "rho Спірмена"],
).round(3)
t_spearman.to_csv(os.path.join(TAB_DIR, "appendix_C_spearman.csv"), index=False, na_rep="—")

t4_article = filter_report.copy()
t4_article["Платформа"] = ["БпЛА", "НРК", "Обидві платформи"]
t4_article.columns = ["Платформа", "Усього SOM", "Відсіяно за S", "Відсіяно за Pb", "Відсіяно за Ne", "Допустимих", "У множині Парето"]
t4_article.to_csv(os.path.join(TAB_DIR, "table4_article_filter.csv"), index=False)

cross_labels = set(cross_feasible["label"])
t5_art_rows = []
for dom_lbl, df_d in [("БпЛА", df_uav), ("НРК", df_ugv)]:
    sel_som = df_d[(df_d["mode"] != "heuristic") & (df_d["pareto"] | df_d["label"].isin(cross_labels))].sort_values(["X", "C"], ascending=[False, True])
    sel_all = pd.concat([sel_som, df_d[df_d["mode"] == "heuristic"]], ignore_index=True)
    for _, r in sel_all.iterrows():
        if r["mode"] == "heuristic":
            st = "Б"
        elif bool(r["pareto"]) and (r["label"] in cross_labels):
            st = "П, К"
        elif bool(r["pareto"]):
            st = "П"
        else:
            st = "К"
        v_s = "—" if pd.isna(r["V"]) else fmt_num(r["V"], 1)
        d_s = "—" if pd.isna(r["D"]) else fmt_num(r["D"], 1)
        lat_s = "—" if pd.isna(r["T50"]) else f"{fmt_num(r['T50'], 2)} / {fmt_num(r['T90'], 2)}"
        u_s = "—" if r["mode"] == "heuristic" else fmt_num(r["U"] / 1000.0, 1)
        t5_art_rows.append((
            dom_lbl, r["label"], st,
            fmt_num(r["X"], 1), f"{int(round(r['Pb']))}", v_s, d_s,
            fmt_num(r["C"], 1), fmt_num(r["Ra"], 1), lat_s,
            fmt_num(r["Ne"], 1), u_s,
        ))

t5_article = pd.DataFrame(
    t5_art_rows,
    columns=["Платформа", "Конфігурація", "Статус", "X, м", "Pb, %", "V, %", "D, м", "C, од.", "Ra, %", "T50 / T90, с", "Ne, од.", "U, тис."],
)
t5_article.to_csv(os.path.join(TAB_DIR, "table5_article_combined.csv"), index=False)

cls_ren = {
    "Швидкі (< 0,35 с)": "Швидкі (до 0,35 с)",
    "Повільні (> 0,55 с)": "Повільні (понад 0,55 с)",
}
t6_art_rows = []
for row in t5_rows:
    dom_lbl, cls_lbl, k_n, x_v, pb_v, v_v, d_v, c_v, s_v, t_v, ra_v, rr_v, rheu_v, rclr_v, t50_v, t90_v, n_v, ne_v, u_v = row
    t6_art_rows.append((
        dom_lbl, cls_ren.get(cls_lbl, cls_lbl), int(k_n),
        fmt_num(x_v, 1), fmt_num(pb_v, 1),
        "—" if pd.isna(v_v) else fmt_num(v_v, 1),
        fmt_num(c_v, 2),
        "—" if pd.isna(s_v) else fmt_num(s_v, 1),
        fmt_num(ra_v, 1),
        "—" if pd.isna(t50_v) else f"{fmt_num(t50_v, 2)} / {fmt_num(t90_v, 2)}",
        "—" if pd.isna(u_v) else fmt_num(u_v, 1),
    ))

t6_article = pd.DataFrame(
    t6_art_rows,
    columns=["Платформа", "Клас / режим", "k", "X, м", "Pb, %", "V, %", "C, од.", "S, %", "Ra, %", "T50 / T90, с", "U, тис."],
)
t6_article.to_csv(os.path.join(TAB_DIR, "table6_article_classes.csv"), index=False)

out_zip = "/content/article_figures_and_tables.zip"
with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder, prefix in [(FIG_DIR, "figures"), (TAB_DIR, "tables")]:
        for fn in sorted(os.listdir(folder)):
            zf.write(os.path.join(folder, fn), f"{prefix}/{fn}")

display(t4_article)
display(t5_article)
display(t6_article)
display(t_spearman)
print("Archive ready:", out_zip)
